# 02_training.ipynb

Train the CNN classifier on encoded transaction images and compare performance with a tabular baseline.

In [12]:
import sys
import subprocess

required_packages = ['pyts', 'imbalanced-learn', 'xgboost']
for package in required_packages:
    try:
        __import__(package)
    except ModuleNotFoundError:
        print(f"Installing missing package: {package}")
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', package])

print('Package check complete.')

Installing missing package: imbalanced-learn
Installing missing package: xgboost
Package check complete.


In [11]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
sys.path.append(str(Path('..').resolve()))
from src.dataset import TransactionImageDataset
from src.model import FraudCNN
from src.train import train_model, evaluate_model, save_checkpoint, create_dataloader

data_path = Path('..') / 'data' / 'transactions_gaf_28.npz'
assert data_path.exists(), 'Run notebook 01_encoding first to generate encoded data.'
dataset = np.load(data_path)
images = dataset['images']
labels = dataset['labels']
train_idx, val_idx = train_test_split(np.arange(len(labels)), test_size=0.2, stratify=labels, random_state=42)
train_images = images[train_idx]
train_labels = labels[train_idx]
val_images = images[val_idx]
val_labels = labels[val_idx]
train_ds = TransactionImageDataset(train_images, train_labels)
val_ds = TransactionImageDataset(val_images, val_labels)
print('Train size:', len(train_ds), 'Val size:', len(val_ds))

Train size: 227845 Val size: 56962


In [5]:
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = FraudCNN(input_channels=1, num_classes=2).to(device)
model, history = train_model(model, train_ds, val_ds, device, batch_size=128, lr=1e-3, num_epochs=10)
save_checkpoint(model, str(Path('..') / 'outputs' / 'models' / 'fraud_cnn_gaf.pth'))

Epoch 1/10: loss=0.4836, f1=0.0000, roc_auc=0.8687, pr_auc=0.0123
Epoch 2/10: loss=0.4607, f1=0.0000, roc_auc=0.8673, pr_auc=0.0122
Epoch 3/10: loss=0.4534, f1=0.0000, roc_auc=0.8682, pr_auc=0.0124
Epoch 4/10: loss=0.4558, f1=0.0000, roc_auc=0.8693, pr_auc=0.0127
Epoch 5/10: loss=0.4568, f1=0.0000, roc_auc=0.8685, pr_auc=0.0127
Epoch 6/10: loss=0.4519, f1=0.0000, roc_auc=0.8737, pr_auc=0.0131
Epoch 7/10: loss=0.4510, f1=0.0000, roc_auc=0.8741, pr_auc=0.0131
Epoch 8/10: loss=0.4536, f1=0.0000, roc_auc=0.8423, pr_auc=0.0123
Epoch 9/10: loss=0.4472, f1=0.0000, roc_auc=0.8754, pr_auc=0.0150
Epoch 10/10: loss=0.4551, f1=0.0000, roc_auc=0.8833, pr_auc=0.0141


In [9]:
val_metrics = evaluate_model(model, create_dataloader(val_ds, batch_size=128, shuffle=False), device)
metrics_df = pd.DataFrame([val_metrics])
metrics_df[['f1', 'precision', 'recall', 'roc_auc', 'pr_auc']]

,f1,precision,recall,roc_auc,pr_auc
0,0.0,0.0,0.0,0.883333,0.014095


## Tabular benchmark (XGBoost)

Train a tabular fraud classifier on the original numeric features for side-by-side comparison.

In [ ]:
import xgboost as xgb
from sklearn.metrics import classification_report, roc_auc_score, average_precision_score
from sklearn.preprocessing import StandardScaler
df = pd.read_csv(Path('..') / 'data' / 'creditcard.csv')
X = df.drop(columns=['Class'])
y = df['Class'].to_numpy()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_val, y_train, y_val = train_test_split(X_scaled, y, test_size=0.2, stratify=y, random_state=42)
clf = xgb.XGBClassifier(eval_metric='logloss', scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum())
clf.fit(X_train, y_train)
y_prob = clf.predict_proba(X_val)[:, 1]
y_pred = clf.predict(X_val)
print('ROC AUC:', roc_auc_score(y_val, y_prob))
print('PR AUC:', average_precision_score(y_val, y_prob))
print(classification_report(y_val, y_pred, digits=4))

c:\Users\palla\myenv\Lib\site-packages\xgboost\training.py:200: UserWarning: [18:42:35] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


ROC AUC: 0.9682378578893572
PR AUC: 0.8800038893818777
              precision    recall  f1-score   support

           0     0.9997    0.9998    0.9998     56864
           1     0.8817    0.8367    0.8586        98

    accuracy                         0.9995     56962
   macro avg     0.9407    0.9183    0.9292     56962
weighted avg     0.9995    0.9995    0.9995     56962

